Libraries

In [72]:
import sys
import os

# بنقول للبايثون: ارجع خطوة لورا ودور هناك كمان
sys.path.append(os.path.abspath('..'))

In [73]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns

Load Data

In [74]:
from src.data_loader import load_data
df=load_data("../data/raw/used_cars.csv")

Solving a problem in the dataset, known through Kaggle discussion on the dataset.


"""

I noticed a little mistake in the data, which probably occured because of the way data was collected or processed after collection.
Car brands that have 2 or more words in the brand name are stored incorrectly. Only the first word is stored as brand name, rest goes into model.

"""

In [75]:
df.head()

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
0,Ford,Utility Police Interceptor Base,2013,"51,000 mi.",E85 Flex Fuel,300.0HP 3.7L V6 Cylinder Engine Flex Fuel Capa...,6-Speed A/T,Black,Black,At least 1 accident or damage reported,Yes,"$10,300"
1,Hyundai,Palisade SEL,2021,"34,742 mi.",Gasoline,3.8L V6 24V GDI DOHC,8-Speed Automatic,Moonlight Cloud,Gray,At least 1 accident or damage reported,Yes,"$38,005"
2,Lexus,RX 350 RX 350,2022,"22,372 mi.",Gasoline,3.5 Liter DOHC,Automatic,Blue,Black,None reported,NaN,"$54,598"
3,INFINITI,Q50 Hybrid Sport,2015,"88,900 mi.",Hybrid,354.0HP 3.5L V6 Cylinder Engine Gas/Electric H...,7-Speed A/T,Black,Black,None reported,Yes,"$15,500"
4,Audi,Q3 45 S line Premium Plus,2021,"9,835 mi.",Gasoline,2.0L I4 16V GDI DOHC Turbo,8-Speed Automatic,Glacier White Metallic,Black,None reported,NaN,"$34,999"


In [76]:
#1-print all unique brands
unique_brands=sorted(df["brand"].unique())
print(unique_brands)

['Acura', 'Alfa', 'Aston', 'Audi', 'BMW', 'Bentley', 'Bugatti', 'Buick', 'Cadillac', 'Chevrolet', 'Chrysler', 'Dodge', 'FIAT', 'Ferrari', 'Ford', 'GMC', 'Genesis', 'Honda', 'Hummer', 'Hyundai', 'INFINITI', 'Jaguar', 'Jeep', 'Karma', 'Kia', 'Lamborghini', 'Land', 'Lexus', 'Lincoln', 'Lotus', 'Lucid', 'MINI', 'Maserati', 'Maybach', 'Mazda', 'McLaren', 'Mercedes-Benz', 'Mercury', 'Mitsubishi', 'Nissan', 'Plymouth', 'Polestar', 'Pontiac', 'Porsche', 'RAM', 'Rivian', 'Rolls-Royce', 'Saab', 'Saturn', 'Scion', 'Subaru', 'Suzuki', 'Tesla', 'Toyota', 'Volkswagen', 'Volvo', 'smart']


In [77]:
#2-create a dictionary for the corrections
corrections={ 
    'Land':'Rover',
    'Alfa':'Romeo',
    'Aston':'Martin',

}

In [78]:
#3-make sure of the noticed columns
for brand,model in corrections.items():
    df_check=df[df['brand']==brand]
    print(df_check.shape[0])
    first_word = df_check['model'].str.split(" ").str[0]
    check=first_word==model
    print(len(check==True))


130
130
19
19
9
9


They are all equal, so here is the problem

In [79]:
#4-solving the problem
def fix_data_integrity(row):
    brand = row['brand']
    model = str(row['model'])
    
    if brand in corrections:
        missing_word = corrections[brand] 
        
        first_word = model.split(" ")[0]

        check = (first_word == missing_word)
        
        if check == True:
            row['brand'] = f"{brand} {missing_word}"
            
            row['model'] = model.replace(missing_word, '', 1).strip()
            
    return row

df = df.apply(fix_data_integrity, axis=1)

print(df[df['brand'].isin(corrections.keys())].head())

Empty DataFrame
Columns: [brand, model, model_year, milage, fuel_type, engine, transmission, ext_col, int_col, accident, clean_title, price]
Index: []


Now,problem is solved

Exploartion

In [80]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4009 entries, 0 to 4008
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   brand         4009 non-null   str  
 1   model         4009 non-null   str  
 2   model_year    4009 non-null   int64
 3   milage        4009 non-null   str  
 4   fuel_type     3839 non-null   str  
 5   engine        4009 non-null   str  
 6   transmission  4009 non-null   str  
 7   ext_col       4009 non-null   str  
 8   int_col       4009 non-null   str  
 9   accident      3896 non-null   str  
 10  clean_title   3413 non-null   str  
 11  price         4009 non-null   str  
dtypes: int64(1), str(11)
memory usage: 376.0 KB


There is a problem in the datatypes:
(price,milage) must be numerical not str

Duplicates

In [81]:
df.duplicated().sum()

np.int64(0)

No duplicates

Data Type Validation

price

In [82]:
df["price"].head()

0    $10,300
1    $38,005
2    $54,598
3    $15,500
4    $34,999
Name: price, dtype: str

In [83]:
df["price"] = df["price"].str.replace('$', '', regex=False).str.replace(',', '', regex=False).astype(int)


In [84]:
df["price"].head()

0    10300
1    38005
2    54598
3    15500
4    34999
Name: price, dtype: int64

milage

In [86]:
df["milage"].head()

0    51,000 mi.
1    34,742 mi.
2    22,372 mi.
3    88,900 mi.
4     9,835 mi.
Name: milage, dtype: str

In [87]:
df["milage"] = df["milage"].str.replace('mi.', '', regex=False).str.replace(',', '', regex=False).astype(int)


In [88]:
df["milage"].head()

0    51000
1    34742
2    22372
3    88900
4     9835
Name: milage, dtype: int64

Descrive statistics

In [89]:
df.describe()

,model_year,milage,price
count,4009.000000,4009.000000,4.009000e+03
mean,2015.515590,64717.551010,4.455319e+04
std,6.104816,52296.599459,7.871064e+04
min,1974.000000,100.000000,2.000000e+03
25%,2012.000000,23044.000000,1.720000e+04
50%,2017.000000,52775.000000,3.100000e+04
75%,2020.000000,94100.000000,4.999000e+04
max,2024.000000,405000.000000,2.954083e+06


Missing Values

In [90]:
print(df.isnull().sum())

brand             0
model             0
model_year        0
milage            0
fuel_type       170
engine            0
transmission      0
ext_col           0
int_col           0
accident        113
clean_title     596
price             0
dtype: int64


fuel_type

In [92]:
fuel_mode = df["fuel_type"].mode()[0]

df["fuel_type"] = df["fuel_type"].fillna(fuel_mode)

In [93]:
print(df["fuel_type"].isnull().sum())

0


accident

In [94]:
accident = df["accident"].mode()[0]

df["accident"] = df["accident"].fillna(fuel_mode)

In [95]:
print(df["accident"].isnull().sum())

0


clean_title

In [96]:
print(df['clean_title'].value_counts())

clean_title
Yes    3413
Name: count, dtype: int64


In [98]:
df['clean_title'] = df['clean_title'].fillna('No')

In [99]:
print(df['clean_title'].value_counts())

clean_title
Yes    3413
No      596
Name: count, dtype: int64


In [100]:
print(df.isnull().sum().sum())

0


No missing values now!!!!!!!!